In [1]:
import numpy as np
import csv
import pandas as pd
from itertools import islice
import json
import matplotlib.pyplot as plt
pd.set_option("display.max_rows", None)


In [2]:
books = pd.read_csv('../data/final_book_dataset_cleaned.csv', sep = '\t',
        dtype={
            'author_birthyear' : 'Int64',
            'title_id' : 'Int64',
            'isbn' : 'str'
        }
    )

In [3]:
books.head(5)

,title_id,title,author,release_year,release_date,first_publisher,author_birthyear,author_birthplace,isbn,book_synopsis,tags,hugo,locus
0,9372,The Long Loud Silence,Wilson Tucker,1954,1954-00-00,Dell,1914,"Deer Creek, Illinois, USA",0899683754,Publisher's description: Tomorrow's war -- the...,"['Anatomy of Wonder 1 Core Collection', 'biolo...",False,False
1,2215181,Mrs. Candy Strikes It Rich,Robert Tallant,1954,1954-00-00,Doubleday,1909,"New Orleans, Louisiana, USA",NaN,NaN,NaN,False,False
2,1392436,Return to the Lost Planet,Angus MacVicar,1954,1954-00-00,Burke,1908,"Duror, Argyll, Scotland, UK",NaN,NaN,NaN,False,False
3,1112206,Rainbow on the Road,Esther Forbes,1954,1954-00-00,Houghton Mifflin,1891,"Westborough, Massachusetts, USA",NaN,NaN,NaN,False,False
4,1908,The Forgotten Planet,Murray Leinster,1954,1954-00-00,Ace Books,1896,"Norfolk, Virginia, USA",0881846163,"**From the first page of the Ace Double:** ""Na...","['insects', 'Librivox', 'Project Gutenberg', '...",False,False


In [4]:
## add month of publication column, 00 means only year is known
books['month_of_publication'] =books['release_date'].str[5:7].str.replace('00','Unknown')
books.sample(50)

,title_id,title,author,release_year,release_date,first_publisher,author_birthyear,author_birthplace,isbn,book_synopsis,tags,hugo,locus,month_of_publication
46493,1267871,Redcoats' Revenge: An Alternate History of the...,"Col. David Fitz-Enz, USA (ret.)",2008,2008-11-30,Potomac Books,<NA>,NaN,9781574889871,NaN,NaN,False,False,11
19229,3210312,"Armageddon, USA",Dan Schmidt,1991,1991-05-00,Bantam Falcon,<NA>,NaN,0553289462,NaN,NaN,False,False,05
25008,871628,The Cold People,Christopher Pike,1996,1996-02-00,Pocket Books,1954,"New York City, New York, USA",0671550640,NaN,NaN,False,False,02
74916,2891611,Child of Lies,Eric Kent Edstrom,2014,2014-08-29,Undermountain Books,<NA>,NaN,9780989901017.0,NaN,NaN,False,False,08
52735,1465521,The Sacred Band,"Janet Morris, Chris Morris",2010,2010-04-05,CreateSpace,1946,"Boston, Massachusetts, USA",9781451599862,NaN,['fantasy'],False,False,04
46960,926620,Vampires of Quentaris,Paul Collins,2008,2008-02-00,Lothian Children's Books / Hachette Australia,1954,"Canvey Island, Essex, England, UK",9780734409850,A synopsis can be found on the [Quentaris\nweb...,['juvenile fantasy'],False,False,02
26355,375751,A Device of Death,Christopher Bulis,1997,1997-02-00,Doctor Who Books,1956,"England, UK",0426205014,The crew of the TARDIS are scattered across th...,"['science fiction', 'English fiction', 'Scienc...",False,False,02
63872,2417011,2013 Evolution,Tosin Coker,2012,2012-12-23,N9neformation,<NA>,NaN,9780955748349,NaN,NaN,False,False,12
125012,3082935,Imitating a Fae Queen,Joanna Reeder,2021,2021-01-16,Reed It & Weep,<NA>,NaN,9798585916564.0,NaN,['young-adult fantasy'],False,False,01
46101,1274363,The Highest Seat Does Not Hold Two,David Williams (IV),2008,2008-09-01,Publication Consultants,<NA>,NaN,9781594330841,NaN,NaN,False,False,09


In [5]:
## Add age of author at publication
books['Author_Age_at_Publication'] = 'Unknown'
books['Author_Age_at_Publication'] = books['Author_Age_at_Publication'].case_when([(books['author_birthyear'].isna() == False,books['release_year'] - books['author_birthyear'])])

In [6]:
books.sample(15)

,title_id,title,author,release_year,release_date,first_publisher,author_birthyear,author_birthplace,isbn,book_synopsis,tags,hugo,locus,month_of_publication,Author_Age_at_Publication
150851,3534971,Midnight Deception,Joline Pearce,2025,2025-07-17,Joline Pearce,<NA>,NaN,9798292933182.0,NaN,"['dark fantasy', 'fantasy romance', 'retold fa...",False,False,07,Unknown
97250,2235726,Pick the Plot,James Riley,2017,2017-09-26,Aladdin,1977,"Connecticut, USA",9781481461283.0,NaN,['juvenile fantasy'],False,False,09,40
147167,3428143,A Flame in the North,John Beresford,2024,2024-07-25,John Beresford,<NA>,NaN,9798334403666.0,NaN,NaN,False,False,07,Unknown
39958,185937,Breathe,Penni Russon,2005,2005-10-00,Greenwillow Books / HarperCollins,1974,"Tasmania, Australia",0060793937,NaN,"['teen fantasy', 'young-adult fantasy']",False,False,10,31
124746,2826290,Death Cultivator 2,Eden Hudson,2020,2020-11-24,Shadow Alley Press,<NA>,NaN,9798587422544.0,NaN,['fantasy'],False,False,11,Unknown
92833,2050521,The Hidden Letters of Velta B.,Gina Ochsner,2016,2016-07-26,Houghton Mifflin Harcourt,1970,USA,9780544253216.0,From a critically acclaimed fiction writer com...,"['Fiction', 'Fantasy', 'General']",False,False,07,46
82963,1903301,After Alice,Gregory Maguire,2015,2015-00-00,Playaway Digital Audio,1954,"Albany, New York, USA",9781467619400.0,NaN,['fantasy'],False,False,Unknown,61
12453,16465,Stitch in Snow,Anne McCaffrey,1984,1984-05-00,Tor,1926,"Cambridge, Massachusetts, USA",0812585623,"Dana Jane Lovell is a savvy, successful, attra...",['Fantasy'],False,False,05,58
91364,2101607,Waylaid,Susan Klaus,2016,2016-11-13,Clay Gulley Publishing,<NA>,"Sarasota, Florida, USA",9780997906400.0,NaN,['fantasy'],False,False,11,Unknown
130469,3059839,The Correction,John Hazen,2021,2021-06-02,Black Rose Writing,<NA>,NaN,9781684337620.0,NaN,['fantasy'],False,False,06,Unknown


In [7]:
## Add column with number of hugo or locus awards won prior to that date
books['Hugo_Awards_Previously']= books.groupby(['author'])['hugo'].cumsum()
books['Locus_Awards_Previously']= books.groupby(['author'])['locus'].cumsum()

In [16]:
display(books[books['author'] == 'Ada Palmer'])
#From the above example we see that .cumsum method counts the current row as well and is causing leakage.

,title_id,title,author,release_year,release_date,first_publisher,author_birthyear,author_birthplace,isbn,book_synopsis,...,hugo,locus,month_of_publication,Author_Age_at_Publication,Hugo_Awards_Previously,Locus_Awards_Previously,Hugo_Nominee_Before,Locus_Nominee_Before,author_birthplace_country,author_birthplace_continent
90307,1989076,Too Like the Lightning,Ada Palmer,2016,2016-05-10,Tor,1981,"Washington, District of Columbia, USA",9780765378002.0,Mycroft Canner is a convict. For his crimes he...,...,True,False,05,35,1,0,True,False,USA,Central/North America
95067,2275090,The Will to Battle,Ada Palmer,2017,2017-12-19,Tor,1981,"Washington, District of Columbia, USA",9780765378040.0,"""The long years of near-utopia have come to an...",...,False,False,12,36,1,0,True,False,USA,Central/North America
100262,2131878,Seven Surrenders,Ada Palmer,2017,2017-02-00,Tor,1981,"Washington, District of Columbia, USA",9780765378026.0,"""It is a world in which near-instantaneous tra...",...,False,True,02,36,1,1,True,True,USA,Central/North America
128775,2923757,Perhaps the Stars,Ada Palmer,2021,2021-10-19,Ad Astra / Head of Zeus,1981,"Washington, District of Columbia, USA",9781786699602.0,The leaders of Hive nations—nations without fi...,...,False,False,10,40,1,1,True,True,USA,Central/North America


In [ ]:
##Once you run this cell the above bug will be fixed.

## Add column with number of hugo or locus awards won prior to that date
## We do this to ensure that hugo info about current year is not included
# cumsum includes the current year as well

books['Hugo_Awards_Previously'] = (
    books.groupby('author')['hugo'].cumsum() - books['hugo']
)
books['Locus_Awards_Previously'] = (
    books.groupby('author')['locus'].cumsum() - books['locus']
)

In [ ]:
display(books[books['author'] == 'Ada Palmer'])
#bug fixed

,title_id,title,author,release_year,release_date,first_publisher,author_birthyear,author_birthplace,isbn,book_synopsis,...,hugo,locus,month_of_publication,Author_Age_at_Publication,Hugo_Awards_Previously,Locus_Awards_Previously,Hugo_Nominee_Before,Locus_Nominee_Before,author_birthplace_country,author_birthplace_continent
90307,1989076,Too Like the Lightning,Ada Palmer,2016,2016-05-10,Tor,1981,"Washington, District of Columbia, USA",9780765378002.0,Mycroft Canner is a convict. For his crimes he...,...,True,False,05,35,0,0,True,False,USA,Central/North America
95067,2275090,The Will to Battle,Ada Palmer,2017,2017-12-19,Tor,1981,"Washington, District of Columbia, USA",9780765378040.0,"""The long years of near-utopia have come to an...",...,False,False,12,36,1,0,True,False,USA,Central/North America
100262,2131878,Seven Surrenders,Ada Palmer,2017,2017-02-00,Tor,1981,"Washington, District of Columbia, USA",9780765378026.0,"""It is a world in which near-instantaneous tra...",...,False,True,02,36,1,0,True,True,USA,Central/North America
128775,2923757,Perhaps the Stars,Ada Palmer,2021,2021-10-19,Ad Astra / Head of Zeus,1981,"Washington, District of Columbia, USA",9781786699602.0,The leaders of Hive nations—nations without fi...,...,False,False,10,40,1,1,True,True,USA,Central/North America


In [19]:
books.sample(15)

,title_id,title,author,release_year,release_date,first_publisher,author_birthyear,author_birthplace,isbn,book_synopsis,...,hugo,locus,month_of_publication,Author_Age_at_Publication,Hugo_Awards_Previously,Locus_Awards_Previously,Hugo_Nominee_Before,Locus_Nominee_Before,author_birthplace_country,author_birthplace_continent
93985,2163763,The Knighthood,Evan Currie,2017,2017-02-08,Evan Currie,<NA>,NaN,NaN,NaN,...,False,False,02,Unknown,0,0,False,False,Unknown,Unknown
95116,2157009,Samaritan,Michael Dempsey (I),2017,2017-01-16,Michael Dempsey (I),<NA>,NaN,9781542386883.0,NaN,...,False,False,01,Unknown,0,0,False,False,Unknown,Unknown
11699,8585,Alanna: The First Adventure,Tamora Pierce,1983,1983-09-00,Beaver Books,1954,"South Connellsville, Pennsylvania, USA",0099435608,NaN,...,False,False,09,29,0,0,False,False,USA,Central/North America
105281,2393774,Yard Full of Bones,"Armand Rosamilia, Jay Wilburn",2018,2018-04-13,Unnerving,<NA>,"New Jersey, USA",9781775254423.0,NaN,...,False,False,04,Unknown,0,0,False,False,USA,Central/North America
130807,3034546,Shaken,Russell Zimmerman,2021,2021-07-21,Catalyst Game Labs,<NA>,NaN,NaN,NaN,...,False,False,07,Unknown,0,0,False,False,Unknown,Unknown
149046,3474208,A Throne of Shadows,Tessonja Odette,2025,2025-04-01,Crystal Moon Press,<NA>,NaN,9781955960311.0,NaN,...,False,False,04,Unknown,0,0,False,False,Unknown,Unknown
114359,2627044,The Darkest Touch,Jaci Burton,2019,2019-05-30,Jaci Burton,<NA>,NaN,9781946535344.0,NaN,...,False,False,05,Unknown,0,0,False,False,Unknown,Unknown
6924,947218,Vicious Spiral,James Ryder,1976,1976-00-00,Robert Hale,1915,"East Harling, Norfolk, England, UK",0709157959,NaN,...,False,False,Unknown,61,0,0,False,False,UK,Europe
54757,1170695,Cryoburn,Lois McMaster Bujold,2010,2010-10-00,Baen Books,1949,"Columbus, Ohio, USA",9781439133941,Kibou-daini is a planet obsessed with cheating...,...,True,True,10,61,9,18,True,True,USA,Central/North America
15806,16213,Dawn for a Distant Earth,"L. E. Modesitt, Jr.",1987,1987-01-00,Tor,1943,"Denver, Colorado, USA",0812516133,"L. E. Modesitt, Jr's first major work was The ...",...,False,False,01,44,0,0,False,False,USA,Central/North America


In [20]:
## Add column with boolean if author been nominated for Hugo/Locus prior to that date
books['Hugo_Nominee_Before'] = 'False'
books['Hugo_Nominee_Before'] = books['Hugo_Nominee_Before'].case_when([(books['Hugo_Awards_Previously'] > 0,True)])
books['Locus_Nominee_Before'] = 'False'
books['Locus_Nominee_Before'] = books['Locus_Nominee_Before'].case_when([(books['Locus_Awards_Previously'] > 0,True)])

In [21]:
books.sample(100)

,title_id,title,author,release_year,release_date,first_publisher,author_birthyear,author_birthplace,isbn,book_synopsis,...,hugo,locus,month_of_publication,Author_Age_at_Publication,Hugo_Awards_Previously,Locus_Awards_Previously,Hugo_Nominee_Before,Locus_Nominee_Before,author_birthplace_country,author_birthplace_continent
99642,2387234,The Wolf's Dream Mate,"Marianne Morea, Milly Taiden",2017,2017-09-12,Coventry Press,<NA>,"New York City, New York, USA",NaN,NaN,...,False,False,09,Unknown,0,0,False,False,USA,Central/North America
57689,1614745,Springman Brothers' Reality Repair,Joshua Wright,2011,2011-00-00,Scholastic Press / Scholastic Australia,<NA>,"Geelong, Victoria, Australia",9781741697858,NaN,...,False,False,Unknown,Unknown,0,0,False,False,Australia,Oceania
5354,18375,The Demon Lover,Dion Fortune,1972,1972-00-00,Aquarian Press,1890,"Bryn-y-Bia, Llandudno, Caernarvonshire, Wales, UK",0850308216,NaN,...,False,False,Unknown,82,0,0,False,False,UK,Europe
132626,3320246,Crown of Lies,Annika West,2022,2022-08-25,Mad Hag Publishing,<NA>,NaN,9798848044034.0,NaN,...,False,False,08,Unknown,0,0,False,False,Unknown,Unknown
97941,2331544,Duchess of Terra,Glynn Stewart,2017,2017-02-10,Faolan's Pen Publishing,<NA>,NaN,9781988035147.0,To preserve humanity's survival and freedom in...,...,False,False,02,Unknown,0,0,False,False,Unknown,Unknown
96150,2715119,Valley of Vengeance,Franklin Horton,2017,2017-06-15,Franklin Horton,<NA>,NaN,9781547192113.0,NaN,...,False,False,06,Unknown,0,0,False,False,Unknown,Unknown
111465,2561810,Betrayal in Time,Julie McElwain,2019,2019-07-02,Pegasus Crime,<NA>,NaN,9781643130743.0,Kendra Donovan’s adventures in nineteenth-cent...,...,False,False,07,Unknown,0,0,False,False,Unknown,Unknown
71979,1529042,Prodigy,Marie Lu,2013,2013-01-29,G. P. Putnam's Sons,1984,"Wuxi, Jiangsu Province, China",9780399256769.0,From copyright page: June and Day make their w...,...,False,False,01,29,0,0,False,False,China,Asia
35554,853216,Wonder Woman: Mythos,Carol Lay,2003,2003-01-00,Pocket Books,1952,"Whittier, California, USA",0743417119,While investigating the bizarre disappearance ...,...,False,False,01,51,0,0,False,False,USA,Central/North America
26019,178278,Body Switchers from Outer Space,"Nina Kiriki Hoffman, R. L. Stine",1996,1996-11-00,Pocket Books,1943,"Columbus, Ohio, USA",0671001868,A humorous tale of an alien who wants to switc...,...,False,False,11,53,0,0,False,False,USA,Central/North America


In [22]:
## Add Author Birthplace by country
replace_dict = {'Austria':['Austria-Hungary','Austro-Hungarian Empire','Austria'],
               'Russia':['Russian Empire','Russia','USSR','Soviet'],
               'Germany':['German Empire','Reich','Confederation','Germany','Prussia'],
               'Nigeria':['Nigeria'],
               'India':['INdia','India'],
               'Peru':['Perú','Peru'],
               'Ceylon':['Ceylon'],
               'Papua New Guinea':['Netherlands New Guinea'],
               'Hong Kong':['Territory','Kong','Hong Kong','Komg'],
               'Dominican Republic':['Dominican Republic'],
               'Bailiwack of Jersey':['Balliwack of Jersey','Bailiwick of Jersey'],
               'Trinidad and Tobago':['Tobago','Trinidad'],
               'UK': ['Yorkshire','Wales','England','United Kingdom','UK','Scotland','Northern Ireland','Guernsey','Britain'],
               'Australia':['Sydney','New South Wales','Australia','Queensland','Victoria','Gold Coast'],
               'USA':['Philadelphia','MO','PA','WA','PR','Baltimore','USA','US Virgin Islands','Puerto Rico'],
               'Panama':['Canal Zone','Panama'],
               'Sri Lanka': ['Lanka'],
               'New Zealand':['Zealand','NZ'],
               'East Indies' :['East Indies'],
                'West Indies' :['West Indies','Antilles','Dominica'],
                'El Salvador' :['El Salvador'],
                'South Africa' :['Cape Colony','Pretoria','Port Elizabeth','South Africa','Orange Free'],
                'United Arab Emirates' :['United Arab Emirates'],
                'Sierra Leone' :['Sierra Leone'],
                'Canada': ['British North America','Canada','Alberta'],
                'Guyana':['Guyana','British Guiana'],
                'Cayman Islands': ['Cayman'],
                'Marshall Islands':['Marshall'],
                'Thailand': ['Thailand','Siam'],
                'Malaysia':['Malaya','Malaysia'],
                'Philipines':['Philipines','Philippine Islands'],
                'Ireland' : ['Ireland','Irish'],
                'Turkey':['Turkey','Ottoman Empire'],
                'Rhodesia':['Rhodesia'],
                'Bermuda':['Bermuda'],
                'Kenya':['Kenya'],
                'Jamaica':['Jamaica'],
                'Sweden':['Sweden','Swedish'],
                'Singapore':['Singapore'],
                'Nyasaland':['Nyasaland'],
                'Palestine':['Palestine'],
                'Fiji':['Fiji'],
                'Holy Roman Empire':['Holy Roman Empire'],
                'Myanmar':['Burma','Myanmar'],
                'Malta':['Malta'],
                'Czech Republic':['Czech'],
                'Rwanda':['Rwanda','Portugese West Africa','Kamundongo']
}

def replace_author_birthplace_country(x):
    if pd.isna(x):
        return 'Unknown'
    for country, birthplace in replace_dict.items():
        if any(word in x for word in birthplace):
            return country
    if x not in replace_dict.items():
        return x.strip().split()[-1]
    return x



In [23]:
books = books.assign(author_birthplace_country=books['author_birthplace'].apply(replace_author_birthplace_country))
books.sample(15)

,title_id,title,author,release_year,release_date,first_publisher,author_birthyear,author_birthplace,isbn,book_synopsis,...,hugo,locus,month_of_publication,Author_Age_at_Publication,Hugo_Awards_Previously,Locus_Awards_Previously,Hugo_Nominee_Before,Locus_Nominee_Before,author_birthplace_country,author_birthplace_continent
108682,2357769,Ravencry,Ed McDonald,2018,2018-06-28,Gollancz,<NA>,NaN,9781473222052.0,"'Dark, twisty and excellent . . . Grimdark wit...",...,False,False,06,Unknown,0,0,False,False,Unknown,Unknown
34818,1150836,Blood of the Tribe,David S. Brody,2003,2003-06-01,Martin and Lawrence Press,<NA>,NaN,0972168710,NaN,...,False,False,06,Unknown,0,0,False,False,Unknown,Unknown
78682,1734437,The Great Day,Robert Reed,2014,2014-02-04,Prime Books,1956,"Omaha, Nebraska, USA",9781607014263.0,NaN,...,False,False,02,58,0,5,False,True,USA,Central/North America
6397,27065,A Stranger Came Ashore,Mollie Hunter,1975,1975-08-07,HarperTrophy,1922,"Longniddry, East Lothian, Scotland, UK",0064400824,"* Set in the Shetland Islands \n * ""Twelve-...",...,False,False,08,53,0,0,False,False,UK,Europe
598,709361,One Against the Moon,Donald A. Wollheim,1956,1956-00-00,World Publishing Co.,1914,"New York City, New York, USA",9781612871523,NaN,...,False,False,Unknown,42,0,0,False,False,USA,Central/North America
35550,985202,Jihad,James Swallow,2003,2003-12-31,Big Finish Productions,1970,"London, England, UK",1844350525,NaN,...,False,False,12,33,0,0,False,False,UK,Europe
106270,2581337,Forever Endangered,Carolina Montague,2018,2018-03-14,The Wild Rose Press,<NA>,NaN,9781509219735.0,NaN,...,False,False,03,Unknown,0,0,False,False,Unknown,Unknown
129454,2838044,Noob Game Plus,Ryan Rimmel,2021,2021-02-22,Ryan Rimmel,<NA>,NaN,9781039400016.0,What happened? The last thing Jim remembered w...,...,False,False,02,Unknown,0,0,False,False,Unknown,Unknown
59544,1249303,Woke Up in a Strange Place,Eric Arvin,2011,2011-02-25,Dreamspinner Press,<NA>,NaN,9781615817955,NaN,...,False,False,02,Unknown,0,0,False,False,Unknown,Unknown
97928,2189937,Shattered Minds,Laura Lam,2017,2017-06-15,Macmillan UK,<NA>,USA,9781447286905.0,Laura Lam's Shattered Minds stars a female 'De...,...,False,False,06,Unknown,0,0,False,False,USA,Central/North America


In [24]:
## Author Birthplace by continent
continent_dict = {
'Africa' : ['Africa','Zambia','Libya','Sudan','Ethiopia','Madagascar','Morocco','Sudan','Namibia','Niger','Botswana','Liberia','Tunisia','Mauritius',
          'Nyasaland','Sierra Leone','Guinea','Tanzania','Zimbabwe','Uganda','Ghana','Egypt','Kenya','Rwanda','Nigeria','Rhodesia','South Africa',
          ],
'Europe' : ['Cyprus','Marino','Macedonia','Latvia','Holy Roman Empire','Luxembourg','Gibraltar','Bulgaria','Ukraine','Serbia','Bailiwack of Jersey',
          'Denmark','Norway','Iceland','Czech Republic','Greece','Hungary','Finland','Spain','Romania','Malta','Belgium','Poland','Portugal',
          'Switzerland','Yugoslavia','Austria','Sweden','Italy','Netherlands','France','Russia','Germany','Ireland','UK'],
'South America' : ['Suriname','Uruguay','Bolivia','Colombia','Guyana','Chile','Ecuador','Peru','Argentina','Brazil','Venezuela'],
'Central/North America' : ['USA','Honduras','Bahamas','Guatemala','Barbados','Cayman Islands','Croatia','Grenada','Panama','Jamaica','Bermuda',
                         'Haiti','Trinidad and Tobago','Cuba','West Indies','Mexico','Dominican Republic','Canada'],
'Asia' : ['Iraq','East Indies','Myanmar','Philippines','Brunei','Macau','Kuwait','Nepal','United Arab Emirates','Bahrain','Vietnam','Laos','Arabia',
        'Pakistan','Palestine','Sri Lanka','Bangladesh','Persia','Turkey','Lebanon','Ceylon','Korea','Thailand','Indonesia','Taiwan','Iran',
        'Singapore','Malaysia','Philippines','Hong Kong','China','Israel','Japan','India'],
'Oceania' : ['Australia','New Zealand','Samoa','Marshall Islands','Papua New Guinea','Fiji']
}

def replace_author_birthplace_continent(x):
    if pd.isna(x):
        return 'Unknown'
    for country, birthplace in continent_dict.items():
        if any(word in x for word in birthplace):
            return country
    if x not in replace_dict.items():
        return x.strip().split()[-1]
    return x

In [25]:
books = books.assign(author_birthplace_continent=books['author_birthplace'].apply(replace_author_birthplace_continent))
books.sample(15)

,title_id,title,author,release_year,release_date,first_publisher,author_birthyear,author_birthplace,isbn,book_synopsis,...,hugo,locus,month_of_publication,Author_Age_at_Publication,Hugo_Awards_Previously,Locus_Awards_Previously,Hugo_Nominee_Before,Locus_Nominee_Before,author_birthplace_country,author_birthplace_continent
146842,3333616,A Cruel Twist of Fate,H. F. Askwith,2024,2024-01-18,Penguin Books,<NA>,NaN,9780241629642.0,And Then There Were None meets The Inheritance...,...,False,False,01,Unknown,0,0,False,False,Unknown,Unknown
98845,2290095,Martha Mayhem and the Witch from the Ditch,Joanne Owen,2017,2017-07-27,Piccadilly Press Ltd,<NA>,"Neyland, Pembrokeshire, Wales, UK",9781848125360.0,NaN,...,False,False,07,Unknown,0,0,False,False,UK,Europe
120314,2776407,The Potion Diaries,Amy McCulloch,2020,2020-10-29,Simon & Schuster Children's UK,1986,"Kingston Upon Thames, London, England, UK",9781471198717.0,Samantha Kemi's ability to mix potions is need...,...,False,False,10,34,0,0,False,False,UK,Europe
119624,2753533,Alien Savage's Stolen Bride,Juno Wells,2020,2020-07-11,Looking Glass Publications,<NA>,NaN,9781948353304.0,NaN,...,False,False,07,Unknown,0,0,False,False,Unknown,Unknown
11204,2060468,Lightfall,Paul Monette,1982,1982-10-00,Avon,1945,"Lawrence, Massachusetts, USA",0380810751,NaN,...,False,False,10,37,0,0,False,False,USA,Central/North America
108167,2371717,Every Time You Go Away,Beth Harbison,2018,2018-07-24,St. Martin's Press,1966,USA,9781250043832.0,In New York Times bestselling author Beth Harb...,...,False,False,07,52,0,0,False,False,USA,Central/North America
68475,2261944,Blood Roses,Lindsay J. Pryor,2013,2013-04-19,Bookouture,<NA>,NaN,9781909490031.0,"""She was supposed to kill vampires, not save t...",...,False,False,04,Unknown,0,0,False,False,Unknown,Unknown
4870,846252,Chalet Diabolique,Virginia Coffman,1971,1971-00-00,Lancer Books,1914,"San Francisco, California, USA",NaN,NaN,...,False,False,Unknown,57,0,0,False,False,USA,Central/North America
131939,3244721,Disciples,"Matthew Peed, Andrew Peed",2022,2022-09-01,Sprocketed Ink,<NA>,NaN,NaN,NaN,...,False,False,09,Unknown,0,0,False,False,Unknown,Unknown
109962,2831223,Iron Dogs,Anthony James,2019,2019-07-09,Anthony James,<NA>,NaN,9781080074839.0,NaN,...,False,False,07,Unknown,0,0,False,False,Unknown,Unknown


In [26]:
books.reset_index()
books.to_csv("data_with_author_and_awards.csv", index = False)